# Pipeline dos Dados

In [ ]:
import pandas as pd
import numpy as np
import scipy.sparse as sp
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from pathlib import Path
import random


Usando dispositivo: cuda


In [5]:
PROCESSED = Path('../../data/processed')
RAW = Path('../../data/raw')

df_edges = pd.read_csv(PROCESSED / 'edges.csv')
df_casos = pd.read_csv(RAW / 'cases.csv')

print(f"Dados reais carregados! Total de arestas com contexto: {len(df_edges)}")
display(df_edges.head())

print(f"Dados de casos carregados! Total de casos: {len(df_casos)}")
display(df_casos.head())

Dados reais carregados! Total de arestas com contexto: 654


,edge_id,case_id,e1_normalizado,e1_text,e1_type,e2_normalizado,e2_text,e2_type,relation_label,intermediate_text
0,E_001,PMC10106591_01,female,female,PATIENT,intermittent chest pain,intermittent chest pain,SYMPTOM,HAS_SYMPTOM,patient presented to the emergency room (ER) o...
1,E_002,PMC10106591_01,physical examination,Physical examination,PROCEDURE,admission,admission,FINDING,SUPPORTS,upon
2,E_003,PMC10106591_01,pulmonary examination,pulmonary examination,PROCEDURE,normal heart,normal heart,FINDING,SUPPORTS,revealed
3,E_004,PMC10106591_01,electrocardiogram,electrocardiogram,PROCEDURE,ecg,ECG,PROCEDURE,SUPPORTS,(
4,E_005,PMC10106591_01,acute coronary syndrome,acute coronary syndrome,DIAGNOSIS,acs,ACS,FINDING,SUPPORTS,(


Dados de casos carregados! Total de casos: 56


,article_id,age,case_id,case_text,gender
0,PMC5137649,44.0,PMC5137649_01,A 44-year-old woman presented with a 3-day his...,Female
1,PMC9387390,57.0,PMC9387390_01,A 57-year-old man with no significant past med...,Male
2,PMC3437073,55.0,PMC3437073_01,A 55-year-old male presented with a gradually ...,Male
3,PMC7102447,65.0,PMC7102447_01,A 65-year-old male with a past medical history...,Male
4,PMC6354154,30.0,PMC6354154_01,A 30-year-old nulligravida presented herself i...,Female


## Preparando Weak Supervision e Gold Standard

In [6]:
df_edges['intermediate_text'] = df_edges['intermediate_text'].fillna('[ADJACENTE]')

casos_forcar_treino = ['PMC5137649_01', 'PMC9387390_01', 'PMC3437073_01', 'PMC7102447_01']

# Lista de todos os casos disponíveis 
casos_unicos = df_casos['case_id'].unique()
casos_para_dividir = [c for c in casos_unicos if c not in casos_forcar_treino]

# Divisão
casos_treino_split, casos_teste = train_test_split(casos_para_dividir, test_size=0.05, random_state=43)

# Adiciona os casos de volta ao treino 
casos_treino = list(casos_treino_split) + casos_forcar_treino

df_treino_ruidoso = df_edges[df_edges['case_id'].isin(casos_treino)].copy()

df_export_casos = df_casos[df_casos['case_id'].isin(casos_teste)].copy()
df_export_casos = df_export_casos[['case_id', 'case_text']]

# Adicionando colunas vazias 
df_export_casos['e1_normalizado'] = ''
df_export_casos['e1_type'] = ''
df_export_casos['e2_normalizado'] = ''
df_export_casos['e2_type'] = ''
df_export_casos['relation_label'] = ''
df_export_casos.to_csv(PROCESSED / 'para_anotar_gold_standard.csv', index=False)

print(f"Casos para Treino (Ruído): {len(casos_treino)} casos -> {len(df_treino_ruidoso)} arestas.")
print(f"Casos para Teste (Gold Standard): {len(casos_teste)} casos brutos exportados para anotação manual.")

Casos para Treino (Ruído): 53 casos -> 620 arestas.
Casos para Teste (Gold Standard): 3 casos brutos exportados para anotação manual.


## TF-IDF e DataLoader PyTorch

In [9]:
vectorizer = TfidfVectorizer(ngram_range=(1, 2), max_features=2000)
X_treino_tfidf = vectorizer.fit_transform(df_treino_ruidoso['intermediate_text'].values)

labels_unicas = df_edges['relation_label'].unique()
label2id = {label: idx for idx, label in enumerate(labels_unicas)}
id2label = {idx: label for label, idx in label2id.items()}

class MLPDataset(Dataset):
    def __init__(self, matriz_tfidf, labels):
        self.x = torch.tensor(matriz_tfidf.toarray(), dtype=torch.float32)
        self.y = torch.tensor([label2id.get(l, 0) for l in labels], dtype=torch.long)
        
    def __len__(self):
        return len(self.x)
        
    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]

treino_dataset = MLPDataset(X_treino_tfidf, df_treino_ruidoso['relation_label'].values)
treino_loader = DataLoader(treino_dataset, batch_size=32, shuffle=True)

print("Tudo pronto! Pipeline de Machine Learning pode iniciar.")
batch_x, batch_y = next(iter(treino_loader))
print(f"Tensores de Entrada X: {batch_x.shape}")
print(f"Tensores de Label Y: {batch_y.shape}")

Tudo pronto! Pipeline de Machine Learning pode iniciar.
Tensores de Entrada X: torch.Size([32, 2000])
Tensores de Label Y: torch.Size([32])


## Arquitetura MLP PyTorch

In [10]:
class RelationExtractorMLP(nn.Module):
    def __init__(self, input_dim, num_classes, hidden_dim=256):
        super(RelationExtractorMLP, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.3)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim // 2)
        self.fc3 = nn.Linear(hidden_dim // 2, num_classes)

    def forward(self, x):
        out = self.fc1(x)
        out = self.relu(out)
        out = self.dropout(out)
        out = self.fc2(out)
        out = self.relu(out)
        out = self.fc3(out)
        return out

input_dimension = X_treino_tfidf.shape[1] # Deve ser max_features (2000)
num_classes = len(label2id)

model = RelationExtractorMLP(input_dim=input_dimension, num_classes=num_classes)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print(model)

RelationExtractorMLP(
  (fc1): Linear(in_features=2000, out_features=256, bias=True)
  (relu): ReLU()
  (dropout): Dropout(p=0.3, inplace=False)
  (fc2): Linear(in_features=256, out_features=128, bias=True)
  (fc3): Linear(in_features=128, out_features=4, bias=True)
)


## Loop de Treinamento

In [ ]:
num_epochs = 1000
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

patience = 50
patience_counter = 0
best_train_loss = float('inf')

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for batch_X, batch_y in treino_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        
        optimizer.zero_grad()
        
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += batch_y.size(0)
        correct += (predicted == batch_y).sum().item()
        
    epoch_loss = running_loss / len(treino_loader)
    epoch_acc = 100 * correct / total

    print(f"Época [{epoch+1}/{num_epochs}], Loss: {epoch_loss:.4f}, Acurácia de Treino: {epoch_acc:.2f}%")

    if epoch_loss < best_train_loss - 0.0001:
        best_train_loss = epoch_loss
        patience_counter = 0
        # Salva o modelo no seu melhor estado de aprendizado
        torch.save(model.state_dict(), PROCESSED / 'best_mlp_relation_extractor.pth')
    else:
        patience_counter += 1
        print(f"  -> Sem melhora na Loss de Treino. Patience: {patience_counter}/{patience}")
        
    if patience_counter >= patience:
        print(f"\nEarly Stopping acionado na época {epoch+1}! O treinamento foi interrompido.")
        break

print("Treinamento concluído!")


Época [1/1000], Loss: 0.2348, Acurácia de Treino: 87.26%
Época [2/1000], Loss: 0.2380, Acurácia de Treino: 87.42%
  -> Sem melhora na Loss de Treino. Patience: 1/50
Época [3/1000], Loss: 0.2404, Acurácia de Treino: 87.42%
  -> Sem melhora na Loss de Treino. Patience: 2/50
Época [4/1000], Loss: 0.2322, Acurácia de Treino: 87.42%
Época [5/1000], Loss: 0.2310, Acurácia de Treino: 86.45%
Época [6/1000], Loss: 0.2381, Acurácia de Treino: 87.26%
  -> Sem melhora na Loss de Treino. Patience: 1/50
Época [7/1000], Loss: 0.2407, Acurácia de Treino: 87.10%
  -> Sem melhora na Loss de Treino. Patience: 2/50
Época [8/1000], Loss: 0.2340, Acurácia de Treino: 87.42%
  -> Sem melhora na Loss de Treino. Patience: 3/50
Época [9/1000], Loss: 0.2340, Acurácia de Treino: 87.58%
  -> Sem melhora na Loss de Treino. Patience: 4/50
Época [10/1000], Loss: 0.2356, Acurácia de Treino: 87.10%
  -> Sem melhora na Loss de Treino. Patience: 5/50
Época [11/1000], Loss: 0.2409, Acurácia de Treino: 87.10%
  -> Sem melho